In [65]:
import os
import time
import pickle
import warnings
import numpy as np
import pandas as pd
import yfinance as yf
import seaborn as sns
import lightgbm as lgb
from yahooquery import Ticker
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

In [66]:
# Suppress warnings
warnings.filterwarnings('ignore')

## Phase 1 — Core System (Base Trading + ML)

In [67]:
# features/feature_engineer.py
class FeatureEngineer:
    def __init__(self):
        pass

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df['range_high'] = df['high'].rolling(window=30).max()
        df['range_low'] = df['low'].rolling(window=30).min()
        df['range_spread'] = df['range_high'] - df['range_low']
        df['body_size'] = (df['close'] - df['open']).abs()
        df['volatility'] = df['high'] - df['low']
        df['volume_spike'] = df['volume'] / df['volume'].rolling(window=30).mean()
        df['avg_volume'] = df['volume'].rolling(window=30).mean()
        df = df.dropna()
        return df

In [68]:
# clustering/breakout_clustering.py
class BreakoutClustering:
    def __init__(self, n_clusters=5):
        self.model = KMeans(n_clusters=n_clusters, random_state=42)

    def fit(self, X):
        self.model.fit(X)

    def predict(self, X):
        return self.model.predict(X)

    def save_model(self, path):
        with open(path, 'wb') as f:
            pickle.dump(self.model, f)

    def load_model(self, path):
        with open(path, 'rb') as f:
            self.model = pickle.load(f)

In [69]:
# labeling/label_creator.py
class LabelCreator:
    def __init__(self, threshold_pct=0.005, max_minutes=30):
        self.threshold_pct = threshold_pct
        self.max_minutes = max_minutes

    def create_labels(self, df):
        labels = []
        for idx in range(len(df) - self.max_minutes):
            entry_price = df.iloc[idx]['close']
            future_prices = df.iloc[idx+1:idx+1+self.max_minutes]['close']
            max_future_return = (future_prices.max() - entry_price) / entry_price
            labels.append(1 if max_future_return >= self.threshold_pct else 0)
        labels += [0]*self.max_minutes
        df['label'] = labels
        return df

In [70]:
# supervised/breakout_classifier.py
class BreakoutClassifier:
    def __init__(self):
        self.model = lgb.LGBMClassifier()

    def fit(self, X, y):
        self.model.fit(X, y)

    def predict(self, X):
        return self.model.predict(X)

    def predict_proba(self, X):
        return self.model.predict_proba(X)[:, 1]

    def save_model(self, path):
        self.model.booster_.save_model(path)

    def load_model(self, path):
        self.model = lgb.Booster(model_file=path)

In [71]:
# trading/portfolio_manager.py
class PortfolioManager:
    def __init__(self, initial_capital=100000, risk_per_trade=0.01):
        self.initial_capital = initial_capital
        self.capital = initial_capital
        self.risk_per_trade = risk_per_trade
        self.positions = []

    def open_position(self, entry_price, stop_loss_price):
        risk_amount = self.capital * self.risk_per_trade
        stop_distance = abs(entry_price - stop_loss_price)
        position_size = risk_amount / stop_distance if stop_distance > 0 else 0
        self.positions.append({'entry_price': entry_price, 'stop_loss': stop_loss_price, 'size': position_size})

    def close_position(self, exit_price, entry_price, size):
        pnl = (exit_price - entry_price) * size
        self.capital += pnl

    def get_capital(self):
        return self.capital

In [72]:
# trading/ml_trading_engine.py
class MLTradingEngine:
    def __init__(self, feature_engineer, cluster_model, classifier_model, portfolio_manager):
        self.feature_engineer = feature_engineer
        self.cluster_model = cluster_model
        self.classifier_model = classifier_model
        self.portfolio_manager = portfolio_manager

    def simulate_trading(self, df):
        df_feat = self.feature_engineer.transform(df)
        cluster_labels = self.cluster_model.predict(df_feat[['range_spread', 'body_size', 'volatility', 'volume_spike']])
        df_feat['cluster'] = cluster_labels

        X = df_feat[['range_spread', 'body_size', 'volatility', 'volume_spike', 'cluster']]
        proba = self.classifier_model.predict_proba(X)

        signals = (proba > 0.6) & (df_feat['cluster'].isin([1, 2]))
        df_feat['signal'] = signals.astype(int)

        equity_curve = [self.portfolio_manager.get_capital()]
        for idx, row in df_feat.iterrows():
            if row['signal'] == 1:
                entry_price = row['close']
                stop_loss = entry_price * 0.98  # Example: 2% stop loss
                self.portfolio_manager.open_position(entry_price, stop_loss)
                self.portfolio_manager.close_position(row['close'], entry_price, self.portfolio_manager.positions[-1]['size'])
            equity_curve.append(self.portfolio_manager.get_capital())

        df_feat['equity'] = equity_curve[:-1]
        return df_feat

## Phase 2 – Building All Plotting + Metrics Modules

In [73]:
# plots/candlestick_plotter.py
class CandlestickPlotter:
    def __init__(self):
        pass

    def plot(self, df, trades_df, save_path=None):
        fig, ax = plt.subplots(figsize=(16, 8))
        df['candle_color'] = df['close'] >= df['open']

        for idx, row in df.iterrows():
            color = 'green' if row['candle_color'] else 'red'
            ax.plot([idx, idx], [row['low'], row['high']], color=color)
            ax.plot([idx, idx], [row['open'], row['close']], color=color, linewidth=6)

        for trade in trades_df.itertuples():
            ax.scatter(trade.entry_time, trade.entry_price, color='blue', marker='^', label='Entry')
            ax.scatter(trade.exit_time, trade.exit_price, color='orange', marker='v', label='Exit')

        ax.xaxis.set_major_locator(mdates.HourLocator(interval=1))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        plt.title('Candlestick Chart with Trades')
        plt.grid(True)
        if save_path:
            plt.savefig(save_path + '/candlestick_trades.png')
        plt.show()

In [74]:
# plots/trading_plotter.py
class TradingPlotter:
    def __init__(self):
        pass

    def plot_equity_curve(self, df, save_path=None):
        plt.figure(figsize=(14, 6))
        plt.plot(df.index, df['equity'])
        plt.title('Equity Curve')
        plt.xlabel('Time')
        plt.ylabel('Portfolio Value')
        plt.grid(True)
        if save_path:
            plt.savefig(save_path + '/equity_curve.png')
        plt.show()

    def plot_drawdowns(self, df, save_path=None):
        rolling_max = df['equity'].cummax()
        drawdown = (df['equity'] - rolling_max) / rolling_max
        plt.figure(figsize=(14, 6))
        plt.fill_between(df.index, drawdown, color='red')
        plt.title('Drawdown Curve')
        plt.xlabel('Time')
        plt.ylabel('Drawdown (%)')
        plt.grid(True)
        if save_path:
            plt.savefig(save_path + '/drawdown_curve.png')
        plt.show()

In [75]:
# plots/trade_distribution_plotter.py
class TradeDistributionPlotter:
    def __init__(self):
        pass

    def plot(self, trades_df, save_path=None):
        returns = (trades_df['exit_price'] - trades_df['entry_price']) / trades_df['entry_price']
        plt.figure(figsize=(10, 6))
        sns.histplot(returns, bins=30, kde=True)
        plt.title('Trade Result Distribution')
        plt.xlabel('Return per Trade')
        plt.grid(True)
        if save_path:
            plt.savefig(save_path + '/trade_result_distribution.png')
        plt.show()

In [76]:
# plots/win_loss_pie_plotter.py
class WinLossPiePlotter:
    def __init__(self):
        pass

    def plot(self, trades_df, save_path=None):
        wins = (trades_df['result'] == 'win').sum()
        losses = (trades_df['result'] == 'loss').sum()
        plt.figure(figsize=(6, 6))
        plt.pie([wins, losses], labels=['Wins', 'Losses'], colors=['green', 'red'], autopct='%1.1f%%')
        plt.title('Win vs Loss Distribution')
        if save_path:
            plt.savefig(save_path + '/win_loss_pie_chart.png')
        plt.show()

In [77]:
# metrics/performance_metrics.py
class PerformanceMetricsCalculator:
    def __init__(self):
        pass

    def calculate(self, trades_df, equity_curve):
        returns = (trades_df['exit_price'] - trades_df['entry_price']) / trades_df['entry_price']
        sharpe_ratio = np.mean(returns) / np.std(returns) * np.sqrt(252) if np.std(returns) != 0 else 0
        win_rate = (trades_df['result'] == 'win').mean() * 100
        profit_factor = returns[returns > 0].sum() / abs(returns[returns < 0].sum()) if returns[returns < 0].sum() != 0 else 0
        max_drawdown = ((equity_curve.cummax() - equity_curve) / equity_curve.cummax()).max()

        metrics = {
            'Sharpe Ratio': sharpe_ratio,
            'Win Rate (%)': win_rate,
            'Profit Factor': profit_factor,
            'Max Drawdown (%)': max_drawdown * 100
        }
        return metrics

In [78]:
# utils/return_calculator.py
class ReturnCalculator:
    def __init__(self):
        pass

    def daily_returns(self, df, save_path=None):
        df['daily_return'] = df['equity'].pct_change()
        daily_returns = df['daily_return'].resample('D').sum()
        plt.figure(figsize=(12, 6))
        daily_returns.plot(kind='bar')
        plt.title('Daily Returns')
        if save_path:
            plt.savefig(save_path + '/daily_returns_plot.png')
        plt.show()

    def monthly_returns(self, df, save_path=None):
        df['monthly_return'] = df['equity'].pct_change()
        monthly_returns = df['monthly_return'].resample('M').sum()
        plt.figure(figsize=(12, 6))
        monthly_returns.plot(kind='bar')
        plt.title('Monthly Returns')
        if save_path:
            plt.savefig(save_path + '/monthly_returns_plot.png')
        plt.show()

In [79]:
# Interval-to-range map
INTERVAL_RANGE_MAP = {
    "1m": "7d",
    "2m": "60d",
    "5m": "60d",
    "15m": "60d",
    "30m": "60d",
    "60m": "730d",
    "90m": "60d",
    "1d": "max",
    "5d": "max",
    "1wk": "max",
    "1mo": "max",
    "3mo": "max"
}

In [80]:
def get_sp500_tickers():
    print("📥 Fetching S&P 500 tickers from Wikipedia...")
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    df = pd.read_html(url, header=0)[0]
    tickers = df['Symbol'].tolist()
    return [ticker.replace('.', '-') for ticker in tickers]


In [81]:
def download_and_save_per_ticker(tickers, interval, data_dir):
    if interval not in INTERVAL_RANGE_MAP:
        raise ValueError(f"Invalid interval: {interval}")

    os.makedirs(data_dir, exist_ok=True)
    total = len(tickers)
    for i, ticker in enumerate(tickers):
        print(f"[{i+1}/{total}] Downloading {ticker}...")
        try:
            t = Ticker(
                ticker,
                asynchronous=False,
                retry=5,
                backoff_factor=0.5,
                status_forcelist=[429, 500, 502, 503, 504],
                timeout=10,
                user_agent="Mozilla/5.0",
                formatted=False
            )
            hist = t.history(interval=interval)
            if isinstance(hist, pd.DataFrame) and not hist.empty:
                hist = hist.reset_index()
                hist["symbol"] = ticker
                hist.to_csv(os.path.join(data_dir, f"{ticker}.csv"), index=False)
        except Exception as e:
            print(f"⚠️ Failed for {ticker}: {e}")
        time.sleep(0.5)  # prevent rate limiting


In [82]:
def concatenate_all(data_dir, output_file):
    print("\n📦 Concatenating all ticker CSVs...")
    all_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith('.csv')]
    df_list = []
    for f in all_files:
        try:
            df = pd.read_csv(f)
            df_list.append(df)
        except Exception as e:
            print(f"⚠️ Could not read {f}: {e}")
    if df_list:
        combined_df = pd.concat(df_list, ignore_index=True)
        combined_df.to_csv(output_file, index=False)
        print(f"\n✅ Combined data saved to: {output_file}")
    else:
        print("\n❌ No data to combine.")



In [ ]:
interval = '60m'
output_folder = f"./data"
final_file = f"./all_sp500_{interval}.csv"

tickers = get_sp500_tickers()
download_and_save_per_ticker(tickers, interval, output_folder)
concatenate_all(output_folder, final_file)

📥 Fetching S&P 500 tickers from Wikipedia...
[1/503] Downloading MMM...
[2/503] Downloading AOS...
[3/503] Downloading ABT...
[4/503] Downloading ABBV...
[5/503] Downloading ACN...
[6/503] Downloading ADBE...
[7/503] Downloading AMD...
[8/503] Downloading AES...
[9/503] Downloading AFL...
[10/503] Downloading A...
[11/503] Downloading APD...
[12/503] Downloading ABNB...
[13/503] Downloading AKAM...


In [15]:
# Step 1: Retrieve the list of S&P 500 companies from Wikipedia
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
sp500_table = pd.read_html(url, header=0)
sp500_df = sp500_table[0]

# Extract the 'Symbol' column
tickers = sp500_df['Symbol'].tolist()

# Step 2: Download historical stock data using yfinance
# Note: Some tickers may have issues (e.g., BRK.B, BF.B), which we'll handle below

# yfinance expects tickers in a specific format; adjust if necessary
# For example, replace '.' with '-' in tickers like 'BRK.B' -> 'BRK-B'
tickers = [ticker.replace('.', '-') for ticker in tickers]

In [16]:
# tickers

In [31]:
t = Ticker(tickers)

In [32]:
t

In [35]:
t.summary_detail

{'MMM': {'maxAge': 1,
  'priceHint': 2,
  'previousClose': 137.93,
  'open': 137.5,
  'dayLow': 136.845,
  'dayHigh': 139.22,
  'regularMarketPreviousClose': 137.93,
  'regularMarketOpen': 137.5,
  'regularMarketDayLow': 136.845,
  'regularMarketDayHigh': 139.22,
  'dividendRate': 2.92,
  'dividendYield': 0.0212,
  'exDividendDate': '2025-02-14 08:00:00',
  'payoutRatio': 0.3524,
  'fiveYearAvgDividendYield': 4.24,
  'beta': 1.06,
  'trailingPE': 17.234123,
  'forwardPE': 17.517721,
  'volume': 1512622,
  'regularMarketVolume': 1512622,
  'averageVolume': 4257995,
  'averageVolume10days': 4307870,
  'averageDailyVolume10Day': 4307870,
  'bid': 0.0,
  'ask': 0.0,
  'bidSize': 900,
  'askSize': 1000,
  'marketCap': 74478870528,
  'fiftyTwoWeekLow': 93.78,
  'fiftyTwoWeekHigh': 156.35,
  'priceToSalesTrailing12Months': 3.0383418,
  'fiftyDayAverage': 143.9404,
  'twoHundredDayAverage': 134.53136,
  'trailingAnnualDividendRate': 2.83,
  'trailingAnnualDividendYield': 0.020517655,
  'curren

In [37]:
historical_data = t.history(interval="60m")

open        high         low  \
symbol date                                                            
MMM    2025-01-02 09:30:00-05:00  130.289993  131.619995  129.160004   
       2025-01-02 10:30:00-05:00  129.250000  130.300003  128.979996   
       2025-01-02 11:30:00-05:00  130.289993  131.070007  130.231201   
       2025-01-02 12:30:00-05:00  130.410004  130.419998  129.860001   
       2025-01-02 13:30:00-05:00  130.089996  130.179993  129.720001   
...                                      ...         ...         ...   
ZTS    2025-04-29 11:30:00-04:00  155.770004  155.895004  155.029999   
       2025-04-29 12:30:00-04:00  155.910004  155.910004  155.320007   
       2025-04-29 13:30:00-04:00  155.725006  156.169998  155.600006   
       2025-04-29 14:30:00-04:00  155.929993  156.220001  155.669998   
       2025-04-29 15:30:00-04:00  155.750000  155.755005  155.220001   

                                       close    volume  dividends  splits  
symbol date                                                                
MMM    2025-01-02 09:30:00-05:00  129.279999  526846.0        0.0     0.0  
       2025-01-02 10:30:00-05:00  130.229996  347732.0        0.0     0.0  
       2025-01-02 11:30:00-05:00  130.399994  270372.0        0.0     0.0  
       2025-01-02 12:30:00-05:00  130.149994  286629.0        0.0     0.0  
       2025-01-02 13:30:00-05:00  130.009995  281321.0        0.0     0.0  
...                                      ...       ...        ...     ...  
ZTS    2025-04-29 11:30:00-04:00  155.820007  197108.0        0.0     0.0  
       2025-04-29 12:30:00-04:00  155.729996  128963.0        0.0     0.0  
       2025-04-29 13:30:00-04:00  155.979996  184012.0        0.0     0.0  
       2025-04-29 14:30:00-04:00  155.740005  238199.0        0.0     0.0  
       2025-04-29 15:30:00-04:00  155.429993  339009.0        0.0     0.0  

[281859 rows x 7 columns]

## Utils Func Manager

In [9]:
# utils/utils_manager.py
class UtilsManager:
    """Unified Utility Manager for FinDRL Project."""

    # ==== Data Loading ====
    @staticmethod
    def load_csv(path: str) -> pd.DataFrame:
        """Load OHLCV data from CSV."""
        df = pd.read_csv(path, parse_dates=['timestamp'])
        df.set_index('timestamp', inplace=True)
        return df

    # ==== Model Saving and Loading ====
    @staticmethod
    def save_pickle(obj, path: str):
        """Save any Python object using pickle."""
        with open(path, 'wb') as f:
            pickle.dump(obj, f)

    @staticmethod
    def load_pickle(path: str):
        """Load any Python object saved with pickle."""
        with open(path, 'rb') as f:
            return pickle.load(f)

    @staticmethod
    def save_lightgbm(model: lgb.Booster, path: str):
        """Save a LightGBM model to file."""
        model.save_model(path)

    @staticmethod
    def load_lightgbm(path: str) -> lgb.Booster:
        """Load a LightGBM model from file."""
        return lgb.Booster(model_file=path)

    # ==== Return Calculations and Plots ====
    @staticmethod
    def plot_daily_returns(df: pd.DataFrame, save_path=None):
        """Plot daily returns from an equity curve."""
        df = df.copy()
        df['daily_return'] = df['equity'].pct_change()
        daily_returns = df['daily_return'].resample('D').sum()

        plt.figure(figsize=(12, 6))
        daily_returns.plot(kind='bar')
        plt.title('Daily Returns')
        plt.xlabel('Date')
        plt.ylabel('Return')
        plt.grid(True)

        if save_path:
            plt.savefig(save_path + '/daily_returns_plot.png')
        plt.show()

    @staticmethod
    def plot_monthly_returns(df: pd.DataFrame, save_path=None):
        """Plot monthly returns from an equity curve."""
        df = df.copy()
        df['monthly_return'] = df['equity'].pct_change()
        monthly_returns = df['monthly_return'].resample('M').sum()

        plt.figure(figsize=(12, 6))
        monthly_returns.plot(kind='bar', color='purple')
        plt.title('Monthly Returns')
        plt.xlabel('Month')
        plt.ylabel('Return')
        plt.grid(True)

        if save_path:
            plt.savefig(save_path + '/monthly_returns_plot.png')
        plt.show()